<a href="https://colab.research.google.com/github/saritrit66-collab/Final-Project/blob/main/Bert_Model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# התקנת הספריות הדרושות בשקט (ללא הדפסת טקסט מיותר)
!pip install transformers sentence-transformers spacy textblob -q
!python -m spacy download en_core_web_sm -q

import pandas as pd
import numpy as np
import spacy
import math
from collections import Counter
from textblob import TextBlob
from transformers import pipeline
from sentence_transformers import SentenceTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report
from sklearn.model_selection import train_test_split
from google.colab import files

print("כל הספריות הותקנו ונטענו בהצלחה!")

✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.
כל הספריות הותקנו ונטענו בהצלחה!


In [ ]:
# 1. טעינת הקובץ המקורי שלך (יש לוודא שהוא מועלה לתיקיית הקבצים בצד שמאל)
# תחליפי את 'full_data.csv' בשם הקובץ האמיתי שלך
df = pd.read_csv('full_data.csv', encoding='utf-8', encoding_errors='ignore')
df['text'] = df['text'].fillna("")

# 2. טעינת מודל השפה לניתוח תחבירי
nlp = spacy.load("en_core_web_sm")

def extract_advanced_features(text):
    text = str(text)
    if not text.strip():
        return pd.Series([0.0, 0.0, 0.0, 0.0, 0.0])

    # תכונות בסיסיות: סנטימנט ואורך מילים
    sentiment = TextBlob(text).sentiment.polarity
    word_count = len(text.split())

    doc = nlp(text)

    # תכונה 1: צפיפות ישויות (NER Density)
    entity_words = sum(len(ent.text.split()) for ent in doc.ents)
    ner_density = entity_words / len(doc) if len(doc) > 0 else 0

    # תכונה 2: יחס תארים לשמות עצם (Adjective to Noun Ratio)
    adjectives = sum(1 for token in doc if token.pos_ == "ADJ")
    nouns = sum(1 for token in doc if token.pos_ == "NOUN")
    adj_noun_ratio = adjectives / nouns if nouns > 0 else 0

    # תכונה 3: אנטרופיית המידע של שנון (Shannon Entropy)
    words = [token.text.lower() for token in doc if token.is_alpha]
    entropy = 0
    if len(words) > 0:
        word_counts = Counter(words)
        for count in word_counts.values():
            p_x = count / len(words)
            entropy -= p_x * math.log2(p_x)

    return pd.Series([sentiment, word_count, ner_density, adj_noun_ratio, entropy])

print("מתחיל לחלץ תכונות מתקדמות (זה ייקח בערך דקה-שתיים)...")
feature_cols = ['Sentiment', 'Word_Count', 'NER_Density', 'Adj_Noun_Ratio', 'Entropy']
df[feature_cols] = df['text'].apply(extract_advanced_features)
print("הנדסת המאפיינים (Feature Engineering) הושלמה!")

מתחיל לחלץ תכונות מתקדמות (זה ייקח בערך דקה-שתיים)...
הנדסת המאפיינים (Feature Engineering) הושלמה!


In [ ]:
# מציאת השורות שצריכות תיוג
mask_unlabeled = df['Is_Conspiracy'].isna()

if mask_unlabeled.any():
    print(f"נמצאו {mask_unlabeled.sum()} שורות ללא תיוג. מתחיל סיווג Zero-Shot...")

    # טעינת מודל סיווג ללא אימון מקדים
    classifier = pipeline("zero-shot-classification", model="facebook/bart-large-mnli", device=0)
    candidate_labels = ["conspiracy theory", "legitimate news"]

    def auto_label(text):
        try:
            res = classifier(str(text), candidate_labels)
            return 1 if res['labels'][0] == "conspiracy theory" else 0
        except:
            return 0 # ברירת מחדל במקרה של שגיאה בטקסט

    # הפעלת הסיווג רק על השורות הריקות
    from tqdm import tqdm
    tqdm.pandas(desc="סיווג רשומות")
    df.loc[mask_unlabeled, 'Is_Conspiracy'] = df.loc[mask_unlabeled, 'text'].progress_apply(auto_label)
    print("תיוג Zero-Shot הושלם בהצלחה!")
else:
    print("כל השורות בדאטה כבר מתויגות.")

נמצאו 19177 שורות ללא תיוג. מתחיל סיווג Zero-Shot...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

...

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

סיווג רשומות: 100%|██████████| 19177/19177 [20:45<00:00, 15.40it/s]

תיוג Zero-Shot הושלם בהצלחה!


In [ ]:
print("טוען מודל BERT לחילוץ מאפיינים סמנטיים (Embeddings)...")
bert_model = SentenceTransformer('all-MiniLM-L6-v2')

# 1. הפיכת הטקסטים ל-384 תכונות מתמטיות בעזרת BERT
X_bert = bert_model.encode(df['text'].tolist(), show_progress_bar=True)

# 2. שילוב עם התכונות הידניות שיצרת
X_custom = df[feature_cols].fillna(0).values
X_combined = np.hstack((X_bert, X_custom))
y = df['Is_Conspiracy'].astype(int).values

# 3. חלוקה לאימון ומבחן
X_train, X_test, y_train, y_test = train_test_split(X_combined, y, test_size=0.2, random_state=42)

# 4. אימון המודל המסכם
print("מאמן את המודל הסופי...")
rf_model = RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=42)
rf_model.fit(X_train, y_train)

# 5. הערכת המודל
y_pred = rf_model.predict(X_test)
print("\n--- תוצאות המודל המשולב ---")
print(classification_report(y_test, y_pred))

# 6. שמירה והורדה של הקובץ המושלם
final_filename = 'Final_Master_Project.csv'
df.to_csv(final_filename, index=False, encoding='utf-8-sig')
print(f"שומר את הקובץ: {final_filename} ומוריד למחשב...")
files.download(final_filename)

טוען מודל BERT לחילוץ מאפיינים סמנטיים (Embeddings)...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/603 [00:00<?, ?it/s]

מאמן את המודל הסופי...

--- תוצאות המודל המשולב ---
              precision    recall  f1-score   support

           0       0.78      1.00      0.88      2990
           1       0.76      0.05      0.09       869

    accuracy                           0.78      3859
   macro avg       0.77      0.52      0.48      3859
weighted avg       0.78      0.78      0.70      3859

שומר את הקובץ: Final_Master_Project.csv ומוריד למחשב...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>